# Slurm

A comprehensive guide to Slurm for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

Slurm (Simple Linux Utility for Resource Management) is a widely used **open-source workload manager** for high-performance computing (HPC) clusters.

### What is it?

- A **cluster scheduler** that manages jobs, queues, and resources (CPUs, GPUs, memory) across nodes.  
- Provides CLI tools (`sbatch`, `squeue`, `srun`, `scancel`, etc.) for submitting and managing jobs.  
- Commonly used in on-prem and cloud-based HPC environments.

### Why use it?

Key benefits of using Slurm:

- **Mature and battle-tested**: Standard scheduler in many academic and enterprise HPC clusters.  
- **Fine-grained resource control**: Request CPUs, GPUs, memory, and other generic resources (GRES).  
- **Job arrays & dependencies**: Express complex batch workloads and pipelines.  
- **Scales to large clusters** with thousands of nodes.

### When to use it?

Slurm is particularly useful when:

- You have or plan to run an **HPC-style cluster** (on-prem or in the cloud).  
- You need tight control over low-level resource allocation (sockets, cores, GPUs, memory).  
- You run **GPU-heavy ML workloads** or simulations that need specialized scheduling policies.

## Key Features

### Core Capabilities of Slurm

| Feature | Description | Benefit |
|--------|-------------|---------|
| **Partitions (queues)** | Logical groupings of nodes with policies. | Separate workloads (debug, gpu, long, etc.). |
| **Resource requests** | `--cpus-per-task`, `--mem`, `--gres=gpu:K`, etc. | Precise resource control per job. |
| **Job arrays** | Submit many similar jobs with indexed tasks. | Hyperparameter sweeps, sharding, simulations. |
| **Job dependencies** | `--dependency=afterok:JOBID` etc. | Build multi-step pipelines. |
| **Accounting & limits** | Fair-share, QoS, and accounting database. | Control usage across users/groups. |
| **Heterogeneous jobs & reservations** | Reserve nodes or define jobs spanning multiple node types. | Support complex workloads and maintenance needs. |

## Architecture Overview

A typical Slurm cluster has:

```text
+----------------------------+
|   Users / CLI clients      |
| (sbatch, squeue, srun)     |
+-------------+--------------+
              |
              v
+----------------------------+
|       slurmctld            |
|   (controller daemon)      |
+-------------+--------------+
              |
              v
+----------------------------+
|   slurmd on each node      |
|   (compute daemons)        |
+----------------------------+
```

### Key components

1. **slurmctld** (controller)  
   - Central scheduler; maintains state and schedules jobs.

2. **slurmd** (node daemon)  
   - Runs on compute nodes; starts/stops job steps.

3. **slurmdbd** (optional)  
   - Accounting database for usage tracking and fair-share.

4. **Partitions**  
   - Group nodes by purpose (e.g., `cpu`, `gpu`, `debug`). Jobs are submitted to partitions.

## Installation

Slurm is typically installed and managed by **cluster administrators**, not by end users or notebooks.

As an end user, you mainly need:

- Access to a cluster where Slurm is already installed.  
- SSH access and the Slurm CLI tools (`sbatch`, `squeue`, `sacct`, `srun`).

For details on installing and configuring Slurm (for cluster admins), see the official documentation links at the end of this notebook.

In [ ]:
# There is no pip-installable "slurm" that gives you a scheduler.
# Slurm is installed on the cluster itself.

print("Use Slurm CLI tools (sbatch, srun, squeue, sacct) on your HPC cluster.")

## Basic Usage

### Submitting a simple job with `sbatch`

You submit jobs by creating a **batch script** with `#SBATCH` directives and then running `sbatch script.sh`.

Example: a CPU-only job:

In [ ]:
# Example Slurm batch script (save as train_cpu.sh)

slurm_script_cpu = """#!/bin/bash
#SBATCH --job-name=cpu-train
#SBATCH --partition=cpu
#SBATCH --time=01:00:00
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G

module load python/3.10

echo "Starting CPU training job on $(hostname)"
python train.py --epochs 10 --device cpu
"""

print(slurm_script_cpu)

# Submit with: sbatch train_cpu.sh

### Requesting GPUs

For ML workloads, you typically request GPUs using GRES:

In [ ]:
# Example Slurm batch script requesting GPUs (save as train_gpu.sh)

slurm_script_gpu = """#!/bin/bash
#SBATCH --job-name=gpu-train
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --time=02:00:00
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G

module load cuda/12.0
module load python/3.10

nvidia-smi

python train.py --epochs 5 --device cuda
"""

print(slurm_script_gpu)

# Submit with: sbatch train_gpu.sh

## Use Cases

- **GPU-accelerated ML training**: Allocate GPUs and CPUs for distributed training jobs.  
- **Hyperparameter search with job arrays**: Submit many small training jobs as a job array.  
- **Large simulations**: Run long-running HPC simulations with precise resource control.  
- **Batch data processing**: Fan out ETL or preprocessing tasks across many nodes.

## Best Practices

1. **Request realistic resources**  
   - Match `--cpus-per-task`, `--mem`, and `--gres` to your workload to avoid wasting resources or being throttled.

2. **Use partitions and QoS correctly**  
   - Submit to the right partition (e.g., `gpu`, `debug`) and observe time limits.

3. **Use job arrays for similar tasks**  
   - Avoid thousands of tiny individual jobs; use `--array` instead.

4. **Leverage modules or virtual environments**  
   - Use environment modules or Conda/venv to manage dependencies.

5. **Automate submission**  
   - Wrap `sbatch` calls in scripts or use higher-level orchestrators (Airflow, Argo, AWS Batch) on top if needed.

## Common Pitfalls

1. **Requesting the wrong partition or resources**  
   - Symptom: Jobs pending forever due to `Resources` or `Priority` reasons.  
   - Fix: Inspect `squeue -j JOBID -o '%i %T %r'` and adjust partition or resource requests.

2. **Ignoring GRES configuration**  
   - Symptom: GPU jobs don’t run or run without visible GPUs.  
   - Fix: Confirm `--gres=gpu:K` matches cluster configuration and that nodes advertise GPUs.

3. **Overloading shared filesystems**  
   - Symptom: Slow job startup due to heavy I/O from many jobs.  
   - Fix: Stage data locally on nodes when possible; stagger starts for massive arrays.

4. **Not cleaning up long-lived jobs**  
   - Symptom: Stale jobs hog resources.  
   - Fix: Use `scancel` and sensible time limits (`--time`).

## Performance Optimization

- **Use job arrays efficiently**:  
  - Submit parameter sweeps as arrays; control max concurrent tasks with `%` (e.g., `--array=0-99%10`).

- **Optimize node utilization**:  
  - Request whole nodes when appropriate; avoid excessive fragmentation of CPUs/GPUs.

- **Use reservations for big jobs**:  
  - Reserve nodes for large multi-node jobs to avoid fragmentation.

- **Profile your code**:  
  - Use profilers to ensure good CPU/GPU utilization inside Slurm jobs.

In [ ]:
# Example: job array snippet (conceptual, save as train_array.sh)

slurm_array = """#!/bin/bash
#SBATCH --job-name=hp-search
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --array=0-9%3

# Use SLURM_ARRAY_TASK_ID to index hyperparameters
python train.py --config configs/config_${SLURM_ARRAY_TASK_ID}.yaml
"""

print(slurm_array)

# Submit with: sbatch train_array.sh

## Production Deployment

- **On-prem clusters**:  
  - Slurm deployed and managed by infrastructure teams; users consume via CLI.

- **Cloud-based HPC**:  
  - Use vendor solutions (e.g., AWS ParallelCluster, Azure CycleCloud, GCP Slurm images) to provision clusters.

- **Multi-tenant environments**:  
  - Use partitions, fair-share, and QoS to separate teams and enforce policies.

- **Integration with higher-level tools**:  
  - Airflow, AWS Batch, and other orchestrators can submit jobs to Slurm as part of larger pipelines.

## Monitoring and Observability

- **Slurm commands**:  
  - `squeue`, `sacct`, `sstat` for job and node status.  

- **Cluster monitoring**:  
  - Use tools like Grafana + Prometheus, Ganglia, or vendor-specific dashboards.

- **Logging**:  
  - Job stdout/stderr are captured to files; centralize them with log shipping if needed.

- **Accounting data**:  
  - Use Slurm accounting DB for usage reports and chargeback/tagging.

## Troubleshooting

- **Jobs pending for a long time**:  
  - Check the `REASON` in `squeue`; may be due to partition limits, resource shortage, or priority.  

- **GPU not visible inside jobs**:  
  - Confirm `nvidia-smi` works; verify `--gres` values and node configuration.  

- **Frequent job failures**:  
  - Inspect job output/error logs; fix application issues or adjust resource requests.

- **Scheduler issues**:  
  - Admins should inspect `slurmctld` logs and cluster health for deeper problems.

## Comparison with Alternatives

| Aspect | Slurm | AWS Batch | Kubernetes-native schedulers |
|--------|-------|----------|------------------------------|
| Hosting | On-prem or cloud VMs | Managed AWS service | Kubernetes cluster |
| Workload type | HPC & batch, arbitrary executables | Containerized batch | Containerized workloads |
| Resource control | Very fine-grained | vCPU/memory, instance types | Pod-level resources |
| Cluster management | You manage | AWS manages EC2 | You manage Kubernetes |

Choose Slurm when you:

- Have or need an **HPC cluster** with fine-grained scheduling.  
- Want a mature scheduler for **large-scale simulations and ML**.  
- Are comfortable managing or using cluster-level infrastructure.

## Resources

- Slurm main page: https://slurm.schedmd.com/  
- Quick start user guide: https://slurm.schedmd.com/quickstart.html  
- `sbatch` documentation: https://slurm.schedmd.com/sbatch.html  
- GRES (GPU) scheduling: https://slurm.schedmd.com/gres.html

These docs cover job submission options, GPU scheduling, accounting, and cluster configuration in detail.